# AB-тестирование часть 2. Стратификация, кластеризация, CUPED/CUPAC

## Стратификация

В обычном A/B-тесте пользователи случайно делятся на группы:

- группа A — контроль

- группа B — эксперимент

Но случайное распределение может привести к перекосу. Например:

- в одной группе окажется больше платящих пользователей

- в другой — больше новых пользователей

- или больше пользователей из другой страны

Это может сильно повлиять на метрики.

Стратификация решает эту проблему:

1) Пользователей делят на страты (однородные сегменты).

2) Внутри каждой страты делают рандомизацию A/B.

| Страта              | A   | B   |
| ------------------- | --- | --- |
| Новые пользователи  | 50% | 50% |
| Старые пользователи | 50% | 50% |
| VIP                 | 50% | 50% |


Делить пользователи по группам можно по следующим признакам:

1) География;
2) История активности;
3) Монетизация;
4) Устройство/платформа.

Также старт **не должно быть слишком много**, а также они должны объяснять **дисперсию целевой метрики**.

А также пользователь должен попасть **только в одну страту**

## Кластеризация

Иногда неизвестно, как распределить на страты лучше. В таких ситуациях помогает **кластеризация**.

Типичный pipeline:

1) берём фичи пользователей

Например:

- число сессий

- средний чек

- число покупок

- время в приложении

- recency

2) применяем алгоритм кластеризации

Часто используют:

- K-means clustering

- Gaussian mixture model

- DBSCAN

3) получаем кластеры пользователей

4) используем кластеры как страты


## Проверка на то, что распределение на страты корректное

Для числовых признаков, например:

- число покупок

- средний чек

- число сессий

- revenue

- lifetime

Используются t-тест (для нормально распределенных значений) и коэффициент Манна-Уитни (U-тест) (для не нормально распределенных значений)

Для проверки категориальных признаков:

- страна;

- устройство;

- тип пользователя.

Используется Chi-square

$$X^2 = \Sigma \frac{(O_i - E_i)^2}{E_i}$$

, где $O_i$ - наблюдаемое явление, $E_i$ - ожидаемое значение

Еще один полезный показатель: стандартизированная средняя разница (SMD) - это статистическая мера размера эффекта, используемая для количественного сравнения разницы между средними значениями двух групп с учётом вариабельности данных.

$$SMD = \frac{\mu_A - \mu_B}{\sigma}$$

### Конечный пайплайн
Хорошая практика:

1) Проверить распределение страт

χ² тест

2) Проверить категориальные признаки

χ² тест

3) Проверить числовые признаки

t-test

Mann-Whitney

4) Посчитать SMD

5) Посмотреть графики распределений

# CUPED и CUPAC как методы борьбы с дисперсией

Методы CUPED и CUPAC — это техники уменьшения дисперсия в A/B-тестировании. 

Их цель простая:
уменьшить дисперсию метрики, чтобы быстрее и надёжнее обнаруживать эффект эксперимента.

Если говорить прямо: они позволяют получить **большую статистическую мощность с тем же числом пользователей**.

В A/B тестировании сравниваются разницы средних двух групп:
$$Δ = \bar{Y_B}-\bar{Y_A}$$

Ошибка выглядит примерно так:
$$SE = \frac{\sigma}{\sqrt{n}}$$

Таким образом, чтобы увеличить мощность теста:
1) увеличить n (дольше ждать)
2) уменьшить σ

## Метод CUPED

Главная идея:

> использовать данные до эксперимента, чтобы объяснить часть вариации метрики.

Представим метрику:

**revenue during experiment**

Но у пользователей есть:

**revenue before experiment**

Обычно они сильно коррелируют.

Например:

| пользователь | revenue_before | revenue_after |
| ------------ | -------------- | ------------- |
| 1            | 10             | 12            |
| 2            | 100            | 110           |
| 3            | 0              | 0             |


Большая часть variance объясняется тем, что одни пользователи всегда тратят больше.

CUPED вычитает этот эффект.

Берём:

Y — метрика в эксперименте

X — метрика до эксперимента

Строим новую метрику:
$$Y' = Y - \theta(X-\hat{X})$$

где $$\theta=\frac{Cov(Y,X)}{Var(X)}$$

| correlation | variance reduction |
| ----------- | ------------------ |
| 0.3         | ~9%                |
| 0.5         | ~25%               |
| 0.7         | ~49%               |


## Алгоритм расчета

Pipeline обычно такой:

1) выбирают pre-period

Например 14 дней до эксперемента

2) считают метрики до эксперемента

Например:

- revenue_before

- sessions_before

- clicks_before

3) считают θ
$$\theta=\frac{Cov(Y,X)}{Var(X)}$$

4) корректируют метрику
$$Y' = Y - \theta(X-\hat{X})$$

5) A/B тестирование делается по новой метрике $Y'$

Интуитивно:

Пользователи имеют разную базовую активность.

Например:
| user | baseline activity |
| ---- | ----------------- |
| A    | высокий           |
| B    | низкий            |

CUPED говорит:

мы знаем baseline → давайте его вычтем.

В результате остаётся только:

Эффект от эксперемента + случайный шум

# CUPAC - улучшенный CUPEC

| метод | covariate                   |
| ----- | --------------------------- |
| CUPED | метрика из прошлого         |
| CUPAC | **ML-предсказание метрики** |


Вместо одного признака X строим модель предсказания метрики.

Например:

модель предсказывает оцениваемую метрику по признакам:

- возраст аккаунта

- число сессий

- device

- регион

- history purchases

В итоге мы получем некоторый прогнозный $\hat{Y}$

И далее строим метрику:

$$Y' = Y - \theta(\hat{Y}-\bar{\hat{Y}})$$

Pipeline для CUPAC:
1) собирают данные пользователей

- количество сессий;

- покупки;

- устройства пользователя;

- география пользователя

- частота использования приложения;

- тип монетизации.

2) обучают модель
например:

- Linear regression;

- Gradient boosting;

- Random forest.

3) получают прогноз $\hat{Y}$;

4) $$Y' = Y - \theta(\hat{Y}-\bar{\hat{Y}})$$